# Cross-slope hydrographic section

A hydrographic section across the Antarctic continental slope,
normal to the 1000 m isobath.

Two pieces of the original are replaced rather than copied:

- the stored `Antarctic_slope_contour_1000m*.nc` files are
  specific to the ACCESS-OM2/OM3 grids, so the contour is
  extracted from the model's own `deptho` using the algorithm of
  `Create_contour.ipynb`;
- `metpy` is not an ESMValTool dependency, so the section is
  sampled by bilinear interpolation along the path between the
  endpoints.

The endpoints are placed as the original places them: from the
chosen contour point, a few degrees inshore along the inward
normal and a few degrees offshore along the outward normal.

---

Translated from the COSIMA recipe
[`Cross-slope_section.ipynb`](https://github.com/COSIMA/cosima-recipes/blob/main/03-Advanced-Recipes/Cross-slope_section.ipynb)
onto CMORised CMIP6 data, using ESMValCore's `Dataset` and
preprocessors in place of the ACCESS-NRI intake catalog.

This notebook is standalone: it needs only `esmvalcore`,
`iris`, `numpy`, `matplotlib`, `cartopy` and `gsw`. The
equivalent ESMValTool recipe is
`recipes/recipe_cosima_*.yml`.

In [ ]:
import numpy as np
import iris
import iris.analysis.cartography
import iris.plot as iplt
import matplotlib.pyplot as plt

from esmvalcore.dataset import Dataset
import gsw
from esmvalcore.preprocessor import (climate_statistics,
                                     extract_region, regrid)

## Start a dask cluster

Every COSIMA recipe opens with one, and these need one for the same
reason, though it is worth being precise about *which* part it speeds up.

**What the cluster does accelerate.** Everything from `Dataset.load()`
through the ESMValCore preprocessors — reading multi-decade files,
`regrid`, `climate_statistics`, `extract_region` — is lazy and chunked.
That is the expensive part of these notebooks and it is what the workers
parallelise. Without a client, dask falls back to its threaded scheduler;
ESMValCore itself warns that the distributed scheduler is preferable.

**What it does not.** Once the preprocessed cube is reduced, the
diagnostic arithmetic realises it into a plain masked array and proceeds
eagerly in numpy. That is deliberate: contour tracing, per-column
remapping and density binning are iterative and index-based, so they do
not express well as array graphs and would be slower, not faster, spread
over workers. Realising *after* the time-mean is what keeps that
affordable — realising a 20-year daily 3-D field before reducing it would
not be.

So: size the cluster for the load-and-preprocess stage. One thread per
worker, as the COSIMA recipes use — with chunks this large, per-worker
memory binds well before thread count does.

On ARE, `Client(threads_per_worker=1)` picks up the cores you requested
for the session. Pass `n_workers=` and `memory_limit=` to be explicit.

**Running the equivalent recipe instead?** Do not start a client there.
ESMValTool manages dask itself, and it defaults to the threaded
scheduler; switch it on in your ESMValTool configuration:

```yaml
dask:
  use: local_distributed
```

In [ ]:
import logging
import os

from dask.distributed import Client

# distributed logs every worker registration and teardown at INFO,
# which is hundreds of lines per notebook and buries everything the
# diagnostic prints. Warnings and errors still come through.
for _name in ("distributed", "distributed.scheduler", "distributed.nanny",
              "distributed.worker", "distributed.core",
              "distributed.batched", "distributed.utils_perf"):
    logging.getLogger(_name).setLevel(logging.WARNING)

# Client.current() reuses a cluster if this cell is re-run, instead of
# quietly starting a second one alongside the first.
try:
    worker_count = max(1, int(os.environ.get("PBS_NCPUS", "1")))
except ValueError:
    worker_count = 1
    print("PBS_NCPUS is not an integer; using one worker")
try:
    client = Client.current()
except ValueError:
    try:
        # Separate worker processes: real memory isolation, and what
        # you want on ARE or any batch node.
        client = Client(n_workers=worker_count, threads_per_worker=1)
    except (RuntimeError, OSError) as error:
        # Some containers and locked-down environments cannot spawn
        # worker subprocesses. In-process workers still give a
        # distributed scheduler and a dashboard, just no isolation.
        print(f'process workers unavailable ({error}); '
              f'using in-process workers instead')
        client = Client(n_workers=worker_count, threads_per_worker=1,
                        processes=False)

active_workers = len(client.scheduler_info()['workers'])
print('dashboard:', client.dashboard_link)
print(f'workers: {active_workers} active (requested {worker_count})')
client

## Load the data

In [ ]:
common = dict(project='CMIP6', exp='historical',
              ensemble='r1i1p1f1', grid='gn',
              dataset='ACCESS-CM2')

thetao = Dataset(short_name='thetao', mip='Omon',
                 timerange='2000/2009', **common)
so = thetao.copy(short_name='so')
deptho = Dataset(short_name='deptho', mip='Ofx', **common)

def southern(cube):
    cube = regrid(cube, target_grid='0.25x0.25', scheme='linear')
    return extract_region(cube, start_longitude=0.,
                          end_longitude=360.,
                          start_latitude=-80., end_latitude=-55.)

thetao_cube = southern(climate_statistics(
    thetao.load(), operator='mean', period='full'))
so_cube = southern(climate_statistics(
    so.load(), operator='mean', period='full'))
deptho_cube = southern(deptho.load())

## Helpers

In [ ]:
# The helpers below are lifted verbatim from
# cosima_cmip/cosima_common.py by notebooks/build_notebooks.py, so
# this notebook runs exactly the same code as the ESMValTool
# diagnostic script. Edit them there, then rebuild.
import contextlib
import warnings

try:  # TEOS-10, needed by the density helpers
    import gsw
except ImportError:
    gsw = None


import logging

LOGGER = logging.getLogger("cosima_cmip")


EARTH_RADIUS = 6371000.0  # m, matches iris.analysis.cartography


FILL_VALUE_THRESHOLD = 1.0e19


def masked_data(cube, dtype=float):
    """Realised, fully masked data of a cube.

    Always go through this rather than ``cube.core_data()``.  The latter
    returns a *lazy* dask array, on which ``numpy.ma`` operations are
    silently no-ops: the mask is not applied, and fill values flow
    straight into the arithmetic.  This realises the data, applies any
    mask the file carries, and additionally masks non-finite values and
    unconverted fill values, which real CMOR files do contain.
    """
    data = np.ma.masked_invalid(np.ma.asarray(cube.data, dtype=dtype))
    return np.ma.masked_where(
        np.abs(np.ma.filled(data, 0.0)) >= FILL_VALUE_THRESHOLD, data)


def _coord_points(cube, name):
    """1D points for ``name``, taking column/row 0 of a 2D aux coord."""
    coord = cube.coord(name)
    if coord.ndim == 1:
        return coord.points
    points = coord.points
    # On a curvilinear grid latitude varies down a column and longitude
    # along a row; take the slice that actually varies.
    if name == "latitude":
        return points[:, 0]
    return points[0, :]


def lat_1d(cube):
    """Representative 1D latitude, also for 2D (curvilinear) coordinates."""
    return _coord_points(cube, "latitude")


def lon_1d(cube):
    """Representative 1D longitude, also for 2D coordinates."""
    return _coord_points(cube, "longitude")


def depth_coord_name(cube):
    """Name of the vertical coordinate, or None for a 2D field."""
    for name in ("depth", "ocean_sigma_z", "olevel", "lev"):
        if cube.coords(name):
            return name
    for coord in cube.coords(dim_coords=True):
        if coord.units.is_convertible("m") and coord.name() != "latitude":
            return coord.name()
    return None


def cell_lengths(cube):
    """Zonal and meridional cell lengths in metres.

    Returns ``(dx, dy)`` broadcast to the horizontal shape of ``cube``.
    ``dx`` shrinks with the cosine of latitude, ``dy`` does not.
    """
    lat = lat_1d(cube)
    lon = lon_1d(cube)
    dlon = np.gradient(lon) * np.pi / 180.0
    dlat = np.gradient(lat) * np.pi / 180.0
    coslat = np.cos(np.deg2rad(lat))
    dx = EARTH_RADIUS * coslat[:, None] * dlon[None, :]
    dy = EARTH_RADIUS * np.broadcast_to(dlat[:, None], dx.shape)
    return dx, np.array(dy)


def horizontal_gradient(field, cube, periodic=True):
    """Central-difference gradient of ``field`` in m-1 units.

    ``field`` is a 2D masked array on the horizontal grid of ``cube``.
    Returns ``(d/dx, d/dy)``.  The zonal direction wraps when
    ``periodic`` is set, which is correct for a global CMOR grid.
    """
    dxm, dym = cell_lengths(cube)
    data = np.ma.masked_invalid(np.ma.asarray(field, dtype=float))
    filled = data.filled(np.nan)

    if periodic:
        padded = np.concatenate(
            [filled[:, -1:], filled, filled[:, :1]], axis=1)
        dfdx = (padded[:, 2:] - padded[:, :-2]) / (2.0 * dxm)
    else:
        dfdx = np.full_like(filled, np.nan)
        dfdx[:, 1:-1] = (filled[:, 2:] - filled[:, :-2]) / (2.0 * dxm[:, 1:-1])

    dfdy = np.full_like(filled, np.nan)
    dfdy[1:-1, :] = (filled[2:, :] - filled[:-2, :]) / (2.0 * dym[1:-1, :])

    return np.ma.masked_invalid(dfdx), np.ma.masked_invalid(dfdy)


def extract_contour(field, level, longest_only=True):
    """Contour of ``field`` at ``level``, in index space.

    ``field`` is a 2D array, typically ``deptho``.  Returns ``(x, y)``
    fractional index coordinates.  As in ``Create_contour.ipynb`` only
    the longest path is kept by default, which discards contours around
    isolated seamounts and depressions.
    """
    import matplotlib

    matplotlib.use("Agg", force=False)
    import matplotlib.pyplot as plt

    data = np.ma.masked_invalid(np.ma.asarray(field, dtype=float))
    figure = plt.figure()
    try:
        axes = figure.add_subplot(111)
        contour_set = axes.contour(np.ma.filled(data, 0.0), levels=[level])
        paths = contour_set.get_paths()
        segments = []
        for path in paths:
            vertices = path.vertices
            if vertices.shape[0] < 2:
                continue
            # A single Path may hold several disjoint polylines; split
            # them where the step between consecutive vertices jumps.
            steps = np.abs(np.diff(vertices[:, 0]).astype(int))
            breaks = np.where(steps > 1)[0]
            start = 0
            for end in list(breaks) + [vertices.shape[0] - 1]:
                if end + 1 - start >= 2:
                    segments.append(vertices[start:end + 1])
                start = end + 1
    finally:
        plt.close(figure)

    if not segments:
        return np.array([]), np.array([])
    if longest_only:
        chosen = max(segments, key=lambda seg: seg.shape[0])
        return chosen[:, 0], chosen[:, 1]
    stacked = np.concatenate(segments, axis=0)
    return stacked[:, 0], stacked[:, 1]


def contour_cells(x_contour, y_contour, shape):
    """Snap a contour to unique, adjacently-connected grid cells.

    Returns ``(rows, cols)`` integer index arrays ordered along the
    contour.  Diagonal steps are filled in with an intermediate cell,
    which ``Cross-contour_transport.ipynb`` requires for the transport
    masks to be exact.
    """
    rows = np.clip(np.rint(y_contour).astype(int), 0, shape[0] - 1)
    cols = np.clip(np.rint(x_contour).astype(int), 0, shape[1] - 1)

    out_rows, out_cols = [], []
    for row, col in zip(rows, cols):
        if out_rows and out_rows[-1] == row and out_cols[-1] == col:
            continue
        if out_rows:
            drow = row - out_rows[-1]
            dcol = col - out_cols[-1]
            if abs(drow) == 1 and abs(dcol) == 1:
                # Insert the cell that makes the step adjacent rather
                # than diagonal; step in x first, as the notebook does.
                out_rows.append(out_rows[-1])
                out_cols.append(col)
        out_rows.append(row)
        out_cols.append(col)
    return np.array(out_rows), np.array(out_cols)


def contour_indices_to_coordinates(rows, cols, lat, lon):
    """Latitude/longitude of integer contour cells."""
    lat = np.asarray(lat)
    lon = np.asarray(lon)
    if lat.ndim == 2:
        return lat[rows, cols], lon[rows, cols]
    return lat[rows], lon[cols]


def topographic_unit_vectors(deptho, cube):
    """Unit vectors tangent and normal to the bathymetry contours.

    Returns ``(t_x, t_y, n_x, n_y)``.  The normal points towards deeper
    water; the tangent is the normal rotated so that shallow water lies
    to the left, matching the sketch in
    ``Along-slope-velocities.ipynb``:

        u_along = u * h_y/|grad h| - v * h_x/|grad h|
        v_cross = u * h_x/|grad h| + v * h_y/|grad h|
    """
    dhdx, dhdy = horizontal_gradient(deptho, cube)
    magnitude = np.ma.sqrt(dhdx**2 + dhdy**2)
    magnitude = np.ma.masked_where(magnitude == 0, magnitude)
    n_x = dhdx / magnitude
    n_y = dhdy / magnitude
    return n_y, -n_x, n_x, n_y


def wrap_longitude(values, reference_lon):
    """Express longitudes in the same convention as ``reference_lon``.

    CMIP6 grids come both as 0-360 and as -180-180, and a section
    longitude written for one is silently wrong on the other: a naive
    ``interp`` clamps everything out of range onto the first column.
    Mapping into ``[lon[0], lon[0] + 360)`` makes the request
    convention-independent.
    """
    reference_lon = np.asarray(reference_lon, dtype=float)
    origin = float(reference_lon[0])
    return origin + (np.asarray(values, dtype=float) - origin) % 360.0


def sample_section(cube, start, end, n_points=60):
    """Bilinearly sample a 3D cube along a straight lon/lat path.

    ``Cross-slope_section.ipynb`` uses
    ``metpy.interpolate.cross_section``; metpy is not an ESMValTool
    dependency, so this does the same job with the grid's own indices.
    The zonal seam wraps, and the longitudes are normalised with
    :func:`wrap_longitude` so the section lands where it was asked for
    whichever convention the grid uses.

    Returns ``(section, lons, lats)`` with ``section`` shaped
    ``(depth, n_points)``.

    The data is realised once, before the per-point loop.  Indexing a
    lazy cube inside the loop makes every point its own dask task and
    ships the whole field to the scheduler again each time, which on
    Gadi produced repeated "Sending large graph of size 33 MiB"
    warnings and dominated the runtime.
    """
    lon = lon_1d(cube)
    lat = lat_1d(cube)
    lons = np.linspace(start[0], end[0], n_points)
    lats = np.linspace(start[1], end[1], n_points)

    data = masked_data(cube)
    if data.ndim == 4:
        data = np.ma.mean(data, axis=0)
    # Realised, contiguous, and never touched lazily again below.
    data = np.ma.masked_array(np.ascontiguousarray(np.ma.filled(data, np.nan)),
                              mask=np.ma.getmaskarray(data))
    data = np.ma.masked_invalid(data)

    col = np.interp(wrap_longitude(lons, lon), lon, np.arange(lon.size))
    row = np.interp(lats, lat, np.arange(lat.size))

    section = np.ma.masked_all((data.shape[0], n_points))
    for point in range(n_points):
        j0 = int(np.floor(row[point]))
        i0 = int(np.floor(col[point]))
        j1 = min(j0 + 1, lat.size - 1)
        i1 = (i0 + 1) % lon.size  # the zonal direction wraps
        weight_j = row[point] - j0
        weight_i = col[point] - i0
        corners = [
            ((1 - weight_j) * (1 - weight_i), data[:, j0, i0]),
            ((1 - weight_j) * weight_i, data[:, j0, i1]),
            (weight_j * (1 - weight_i), data[:, j1, i0]),
            (weight_j * weight_i, data[:, j1, i1]),
        ]
        accumulated = np.ma.zeros(data.shape[0])
        weights = np.zeros(data.shape[0])
        for weight, values in corners:
            valid = ~np.ma.getmaskarray(values)
            accumulated[valid] += weight * values[valid]
            weights[valid] += weight
        section[:, point] = np.ma.masked_where(
            weights == 0, accumulated / np.where(weights == 0, 1.0, weights))
    return section, lons, lats


def report_section_coverage(section, label="section"):
    """Log how much of a sampled section actually has water.

    A section that misses the ocean -- wrong endpoints, wrong longitude
    convention, a path over land -- produces an all-masked array, and a
    blank figure looks much like a failed one.  Saying so up front is
    the difference between a bug and a result.
    """
    total = section.size
    valid = int(np.ma.count(section))
    if valid == 0:
        LOGGER.warning(
            "the %s is entirely masked: no ocean along that path. Check "
            "the endpoints and that the section longitude is inside the "
            "extracted region.", label)
    elif valid < 0.1 * total:
        LOGGER.warning(
            "only %d of %d %s points have data (%.0f%%); the path is "
            "mostly over land or outside the region",
            valid, total, label, 100 * valid / total)
    else:
        LOGGER.info("%s has data at %d of %d points (%.0f%%)",
                    label, valid, total, 100 * valid / total)
    return valid


@contextlib.contextmanager
def quiet_gsw():
    """Silence gsw's masked-array warning for the enclosed call.

    ``gsw`` emits ``'where' used without 'out', expect uninitialized
    memory in output`` every time it is handed a masked array, which is
    every time here.  The uninitialised values sit only underneath the
    mask and never reach the result -- asserted in
    ``validation/test_cosima_common.py`` -- but the warning fires once
    per call and buries everything else a notebook prints.

    Scoped to this one message so genuine gsw warnings still surface.
    """
    with warnings.catch_warnings():
        warnings.filterwarnings(
            "ignore", message=".*'where' used without 'out'.*",
            category=UserWarning)
        yield


def _require_gsw():
    if gsw is None:
        raise ImportError(
            "gsw (TEOS-10) is required for the density diagnostics; "
            "install it with `pixi add gsw`")


def pressure_from_depth(depth, lat):
    """Sea pressure in dbar from depth in metres (positive down)."""
    _require_gsw()
    depth = np.asarray(depth, dtype=float)
    lat = np.asarray(lat, dtype=float)
    return gsw.p_from_z(-depth, lat)


def absolute_salinity(practical_salinity, pressure, lon, lat):
    """Practical salinity (PSU) to absolute salinity (g/kg)."""
    _require_gsw()
    with quiet_gsw():
        return gsw.SA_from_SP(practical_salinity, pressure, lon, lat)


def conservative_temperature(abs_salinity, potential_temperature):
    """Potential temperature (degC) to conservative temperature (degC)."""
    _require_gsw()
    with quiet_gsw():
        return gsw.CT_from_pt(abs_salinity, potential_temperature)


def _broadcast_to_field(depth, lat, lon, shape):
    """Broadcast depth/lat/lon onto a ``(depth, lat, lon)`` field shape."""
    depth_b = np.broadcast_to(np.asarray(depth)[:, None, None], shape)
    lat_b = np.broadcast_to(np.asarray(lat)[None, :, None], shape)
    lon_b = np.broadcast_to(np.asarray(lon)[None, None, :], shape)
    return depth_b, lat_b, lon_b


def teos10_fields(thetao_cube, so_cube, temperature_is_conservative=False):
    """Absolute salinity, conservative temperature and pressure.

    Accepts fields with optional leading dimensions, such as time:
    ``(lat, lon)``, ``(depth, lat, lon)`` or
    ``(time, depth, lat, lon)``.  It returns masked arrays broadcast to
    the cube shape.  CMIP ``thetao`` is potential temperature, so it is
    converted unless the caller says otherwise (MOM5 ``temp`` is already
    conservative).
    """
    _require_gsw()
    salinity = masked_data(so_cube)
    temperature = masked_data(thetao_cube)

    if salinity.shape != temperature.shape:
        raise ValueError("thetao and so must have the same shape")

    def broadcast_1d(points, axis):
        """Broadcast a coordinate along its corresponding data axis."""
        reshape = [1] * salinity.ndim
        reshape[axis] = len(points)
        return np.broadcast_to(np.asarray(points).reshape(reshape),
                               salinity.shape)

    lat = lat_1d(so_cube)
    lon = lon_1d(so_cube)
    lat_axis = so_cube.coord_dims("latitude")[0]
    lon_axis = so_cube.coord_dims("longitude")[-1]
    depth_name = depth_coord_name(so_cube)
    if depth_name is None:
        depth_b = np.zeros(salinity.shape)
    else:
        depth = so_cube.coord(depth_name).points
        depth_axis = so_cube.coord_dims(depth_name)[0]
        depth_b = broadcast_1d(depth, depth_axis)
    lat_b = broadcast_1d(lat, lat_axis)
    lon_b = broadcast_1d(lon, lon_axis)
    pressure = pressure_from_depth(depth_b, lat_b)
    abs_sal = absolute_salinity(salinity, pressure, lon_b, lat_b)
    if temperature_is_conservative:
        cons_temp = temperature
    else:
        cons_temp = conservative_temperature(abs_sal, temperature)

    return abs_sal, cons_temp, pressure


def potential_density_anomaly(thetao_cube, so_cube, p_ref=2000,
                              temperature_is_conservative=False):
    """Potential density anomaly sigma_n in kg m-3.

    ``p_ref`` is the reference pressure in dbar; 0, 1000, 2000, 3000 and
    4000 map onto the ``gsw.sigma0``...``gsw.sigma4`` family.
    """
    _require_gsw()
    sigma_functions = {
        0: gsw.sigma0,
        1000: gsw.sigma1,
        2000: gsw.sigma2,
        3000: gsw.sigma3,
        4000: gsw.sigma4,
    }
    if p_ref not in sigma_functions:
        raise ValueError(
            f"p_ref must be one of {sorted(sigma_functions)}, got {p_ref}")
    abs_sal, cons_temp, _ = teos10_fields(
        thetao_cube, so_cube,
        temperature_is_conservative=temperature_is_conservative)
    with quiet_gsw():
        return np.ma.masked_invalid(
            sigma_functions[p_ref](abs_sal, cons_temp))

## Find the section endpoints

In [ ]:
CONTOUR_DEPTH = 1000.0
SECTION_LONGITUDE = 300.0
SHELF_DISTANCE, OCEAN_DISTANCE = 3.0, 5.0

depth = masked_data(deptho_cube)
x_contour, y_contour = extract_contour(
    np.ma.filled(depth, 0.0), CONTOUR_DEPTH)
rows, cols = contour_cells(x_contour, y_contour, depth.shape)

lat, lon = lat_1d(deptho_cube), lon_1d(deptho_cube)
contour_lat, contour_lon = contour_indices_to_coordinates(
    rows, cols, lat, lon)

offset = np.abs(((contour_lon - SECTION_LONGITUDE + 180.0) % 360.0)
                - 180.0)
index = int(np.argmin(offset))
point_lon = float(contour_lon[index])
point_lat = float(contour_lat[index])

_, _, n_x, n_y = topographic_unit_vectors(depth, deptho_cube)
nx = float(np.ma.filled(n_x[rows[index], cols[index]], 0.0))
ny = float(np.ma.filled(n_y[rows[index], cols[index]], -1.0))
norm = np.hypot(nx, ny) or 1.0
nx, ny = nx / norm, ny / norm

start = (point_lon - SHELF_DISTANCE * nx,
         point_lat - SHELF_DISTANCE * ny)
end = (point_lon + OCEAN_DISTANCE * nx,
       point_lat + OCEAN_DISTANCE * ny)
print(f'section from {start} to {end}')

## Sample the section

`metpy.interpolate.cross_section` does this upstream; metpy is
not an ESMValTool dependency, so `sample_section` above does the
same job with the grid's own indices.

One trap worth naming, because it is silent. CMIP6 grids come
both as 0-360 and as -180-180, and a section longitude written
for one convention lands nowhere on the other: interpolating a
longitude that falls outside the axis simply **clamps** to the
first column, so asking for 300E on a 0-360 grid quietly returns
a section at 0.5E. `wrap_longitude` normalises the request into
whatever convention the grid uses, and the zonal seam wraps
rather than clamping. Since this notebook works on a regridded
(regular) grid, that is all that is needed.

In [ ]:
N_POINTS = 60

temperature, lons, lats = sample_section(thetao_cube, start, end,
                                         n_points=N_POINTS)

sigma2 = potential_density_anomaly(thetao_cube, so_cube,
                                   p_ref=2000)
sigma_cube = thetao_cube.copy(np.ma.filled(sigma2, np.nan))
sigma_section, _, _ = sample_section(sigma_cube, start, end,
                                     n_points=N_POINTS)

depth_levels = thetao_cube.coord(
    depth_coord_name(thetao_cube)).points

# A blank figure and a failed one look identical; say which.
report_section_coverage(temperature, 'temperature section')
report_section_coverage(sigma_section, 'density section')

# A distance coordinate is monotonic even when the section is
# nearly zonal, unlike latitude.  It also puts 0 at the isobath,
# as in the original COSIMA figure.
dlon = np.deg2rad((lons - point_lon + 180.0) % 360.0 - 180.0)
dlat = np.deg2rad(lats - point_lat)
distance = EARTH_RADIUS * np.sqrt(
    dlat**2 + (np.cos(np.deg2rad(point_lat)) * dlon)**2) / 1.e3
direction = np.linspace(-SHELF_DISTANCE, OCEAN_DISTANCE,
                        N_POINTS)
distance = np.copysign(distance, direction)

## Plot

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

mesh = ax.pcolormesh(distance, depth_levels, temperature,
                     cmap='RdYlBu_r', shading='auto')
if np.ma.count(sigma_section):
    contours = ax.contour(distance, depth_levels, sigma_section,
                          colors='k', linewidths=0.7, levels=10)
    ax.clabel(contours, inline=True, fontsize=7, fmt='%.2f')

ax.invert_yaxis()
ax.axvline(0, color='k', linewidth=0.6)
ax.set_xlabel('Distance from 1000 m isobath (km)')
ax.set_ylabel('Depth (m)')
ax.set_title(f'Cross-slope section through the '
             f'{CONTOUR_DEPTH:.0f} m isobath')
fig.colorbar(mesh, ax=ax, shrink=0.85,
             label='Potential temperature ($^\circ$C)')
plt.tight_layout()
fig                       # always emit the figure in Jupyter

## Shut the cluster down

Workers hold their memory until the cluster closes. Leaving one running
in an idle kernel is the usual reason a later notebook cannot get the
memory it asks for.

In [ ]:
client.close()